# Congressional Trading Risk Intelligence
## 01 — Data Cleaning & Preparation

This notebook prepares U.S. congressional stock-disclosure data for later analysis of trading patterns, reporting timelines, sector exposure, and potential ethics/compliance indicators.

### Objectives
- inspect the raw dataset;
- standardize column names and text fields;
- parse dates;
- clean ticker and transaction fields;
- convert disclosed amount ranges into numeric features;
- calculate disclosure timing where possible;
- identify and remove exact duplicate records; and
- export a reproducible cleaned dataset.

> **Important:** This project analyzes public financial-disclosure data. It does not establish insider trading, unlawful conduct, or an ethics violation by any lawmaker.


## 1. Import libraries

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)


## 2. Load the raw data

In [2]:
DATA_PATH = Path("../data/raw/US_CONGRESS_DATA.csv")

raw = pd.read_csv(DATA_PATH)

print(f"Rows: {len(raw):,}")
print(f"Columns: {raw.shape[1]}")
raw.head()


Rows: 17,170
Columns: 16


,disclosure_year,disclosure_date,transaction_date,owner,ticker,asset_description,type,amount,representative,district,state,ptr_link,cap_gains_over_200_usd,industry,sector,party
0,2021,10/04/2021,2021-09-27,joint,BP,BP plc,purchase,"$1,001 - $15,000",Virginia Foxx,NC05,NC,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2021/20019557.pdf,False,Integrated oil Companies,Energy,Republican
1,2021,10/04/2021,2021-09-13,joint,XOM,Exxon Mobil Corporation,purchase,"$1,001 - $15,000",Virginia Foxx,NC05,NC,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2021/20019557.pdf,False,Integrated oil Companies,Energy,Republican
2,2021,10/04/2021,2021-09-10,joint,ILPT,Industrial Logistics Properties Trust - Common Shares of Beneficial Interest,purchase,"$15,001 - $50,000",Virginia Foxx,NC05,NC,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2021/20019557.pdf,False,Real Estate Investment Trusts,Real Estate,Republican
3,2021,10/04/2021,2021-09-28,joint,PM,Phillip Morris International Inc,purchase,"$15,001 - $50,000",Virginia Foxx,NC05,NC,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2021/20019557.pdf,False,Farming/Seeds/Milling,Consumer Non-Durables,Republican
4,2021,10/04/2021,2021-09-17,self,BLK,BlackRock Inc,sale_partial,"$1,001 - $15,000",Alan S. Lowenthal,CA47,CA,https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2021/20019570.pdf,False,Investment Bankers/Brokers/Service,Finance,Democrat


## 3. Inspect structure and missing values

In [3]:
raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17170 entries, 0 to 17169
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   disclosure_year         17170 non-null  int64 
 1   disclosure_date         17170 non-null  object
 2   transaction_date        17170 non-null  object
 3   owner                   10524 non-null  object
 4   ticker                  17170 non-null  object
 5   asset_description       17166 non-null  object
 6   type                    17170 non-null  object
 7   amount                  17170 non-null  object
 8   representative          17170 non-null  object
 9   district                17170 non-null  object
 10  state                   17170 non-null  object
 11  ptr_link                17170 non-null  object
 12  cap_gains_over_200_usd  17170 non-null  bool  
 13  industry                12421 non-null  object
 14  sector                  12421 non-null  object
 15  pa

In [4]:
missing_report = (
    raw.isna().sum()
    .sort_values(ascending=False)
    .rename("Missing_Values")
    .to_frame()
)

missing_report["Missing_Percent"] = (
    missing_report["Missing_Values"] / len(raw) * 100
).round(1)

missing_report


,Missing_Values,Missing_Percent
owner,6646,38.7
sector,4749,27.7
industry,4749,27.7
party,85,0.5
asset_description,4,0.0
transaction_date,0,0.0
disclosure_date,0,0.0
disclosure_year,0,0.0
amount,0,0.0
type,0,0.0


## 4. Standardize column names

In [5]:
df = raw.copy()

df.columns = [
    re.sub(r"\s+", "_", column.strip().lower())
    for column in df.columns
]

df.columns.tolist()


['disclosure_year',
 'disclosure_date',
 'transaction_date',
 'owner',
 'ticker',
 'asset_description',
 'type',
 'amount',
 'representative',
 'district',
 'state',
 'ptr_link',
 'cap_gains_over_200_usd',
 'industry',
 'sector',
 'party']

## 5. Parse date columns

In [6]:
date_columns = [c for c in df.columns if "date" in c]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

df[date_columns].head()


,disclosure_date,transaction_date
0,2021-10-04,2021-09-27
1,2021-10-04,2021-09-13
2,2021-10-04,2021-09-10
3,2021-10-04,2021-09-28
4,2021-10-04,2021-09-17


## 6. Clean text fields

In [7]:
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

text_columns.tolist()


['owner',
 'ticker',
 'asset_description',
 'type',
 'amount',
 'representative',
 'district',
 'state',
 'ptr_link',
 'industry',
 'sector',
 'party']

## 7. Standardize tickers and transaction types

In [8]:
ticker_columns = [c for c in df.columns if "ticker" in c]

for column in ticker_columns:
    df[column] = df[column].replace({
        "--": pd.NA,
        "N/A": pd.NA,
        "nan": pd.NA,
        "None": pd.NA
    })

transaction_type_columns = [
    c for c in df.columns
    if "transaction" in c and "type" in c
]

for column in transaction_type_columns:
    df[column] = df[column].str.title()

print("Ticker columns:", ticker_columns)
print("Transaction type columns:", transaction_type_columns)


Ticker columns: ['ticker']
Transaction type columns: []


## 8. Convert disclosed amount ranges into numeric features

Congressional disclosures often report transactions as ranges rather than exact values. To preserve that uncertainty, the notebook creates:

- `amount_min`
- `amount_max`
- `amount_midpoint`

The midpoint is a modelling convenience for aggregate analysis. It should not be interpreted as the actual transaction value.


In [9]:
amount_columns = [c for c in df.columns if "amount" in c]
amount_col = amount_columns[0] if amount_columns else None

def parse_amount_bounds(value):
    if pd.isna(value):
        return np.nan, np.nan, np.nan

    text = str(value).replace("$", "").replace(",", "").strip()
    numbers = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", text)]

    if not numbers:
        return np.nan, np.nan, np.nan

    if len(numbers) == 1:
        lower = upper = numbers[0]
    else:
        lower, upper = numbers[0], numbers[1]

    return lower, upper, (lower + upper) / 2

if amount_col:
    parsed = df[amount_col].apply(parse_amount_bounds)

    df["amount_min"] = [x[0] for x in parsed]
    df["amount_max"] = [x[1] for x in parsed]
    df["amount_midpoint"] = [x[2] for x in parsed]

    df[[amount_col, "amount_min", "amount_max", "amount_midpoint"]].head(10)
else:
    print("No amount column found.")


## 9. Calculate disclosure lag

If both transaction and disclosure dates are present, the difference between them is calculated as `disclosure_lag_days`.

This is only a descriptive timing feature at this stage. Whether a disclosure is legally late depends on the applicable statutory rules and factual context, which should be analyzed separately.


In [10]:
transaction_date_col = (
    "transaction_date" if "transaction_date" in df.columns else None
)
disclosure_date_col = (
    "disclosure_date" if "disclosure_date" in df.columns else None
)

if transaction_date_col and disclosure_date_col:
    df["disclosure_lag_days"] = (
        df[disclosure_date_col] - df[transaction_date_col]
    ).dt.days

    df[
        [
            transaction_date_col,
            disclosure_date_col,
            "disclosure_lag_days"
        ]
    ].head(10)
else:
    print("Transaction and disclosure date columns were not both found.")


## 10. Check duplicates

In [11]:
exact_duplicates = df.duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicates:,}")


Exact duplicate rows: 554


In [12]:
clean = df.drop_duplicates().reset_index(drop=True)

print(f"Rows before deduplication: {len(df):,}")
print(f"Rows after deduplication:  {len(clean):,}")


Rows before deduplication: 17,170
Rows after deduplication:  16,616


## 11. Final validation

In [13]:
validation = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Exact duplicates",
        "Missing transaction dates" if "transaction_date" in clean.columns else "Missing transaction dates",
        "Missing disclosure dates" if "disclosure_date" in clean.columns else "Missing disclosure dates",
        "Unique lawmakers" if "representative" in clean.columns else "Unique names"
    ],
    "Value": [
        len(clean),
        clean.shape[1],
        clean.duplicated().sum(),
        clean["transaction_date"].isna().sum() if "transaction_date" in clean.columns else np.nan,
        clean["disclosure_date"].isna().sum() if "disclosure_date" in clean.columns else np.nan,
        clean["representative"].nunique() if "representative" in clean.columns else np.nan
    ]
})

validation


,Metric,Value
0,Rows,16616
1,Columns,20
2,Exact duplicates,0
3,Missing transaction dates,12
4,Missing disclosure dates,0
5,Unique lawmakers,196


## 12. Export cleaned data

In [14]:
OUTPUT_PATH = Path("../data/processed/us_congress_trades_clean.csv")

clean.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Cleaned rows: {len(clean):,}")


Saved: ../data/processed/us_congress_trades_clean.csv
Cleaned rows: 16,616


## Cleaning summary

The dataset is now prepared for exploratory analysis.

### Key transformations
- standardized column names;
- parsed date fields;
- normalized whitespace and text fields;
- cleaned ticker placeholders;
- standardized transaction-type text;
- converted disclosed amount ranges into numeric lower, upper, and midpoint values;
- calculated transaction-to-disclosure timing where possible;
- removed exact duplicate rows.

### Next notebook

**02 — Trading Patterns Analysis**

The next stage will examine:
- transaction types;
- trading activity by lawmaker;
- sector and industry exposure;
- transaction-size ranges;
- buy versus sell patterns; and
- trading activity over time.

A later notebook can then analyze disclosure timing against the relevant congressional financial-disclosure rules.
